# LeetCode 2064 — Minimized Maximum of Products Distributed to Any Store bằng Python

Link đề: https://leetcode.com/problems/minimized-maximum-of-products-distributed-to-any-store/ · Độ khó: **Medium** · Chủ đề: Array, Binary Search, Greedy

Skeleton LeetCode:

```python
class Solution:
    def minimizedMaximum(self, n: int, quantities: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

1. Nhận dạng bài "**cực tiểu hóa giá trị lớn nhất**" ⇒ **binary search trên đáp án** với hàm kiểm tra đơn điệu.
2. Hàm kiểm tra: với ngưỡng `x`, loại `i` cần **ít nhất** `⌈q_i / x⌉` cửa hàng ⇒ khả thi ⇔ `Σ ⌈q_i / x⌉ ≤ n`.
3. Cách tham lam tương đương bằng **heap**: liên tục cấp thêm cửa hàng cho loại đang có "tải mỗi cửa hàng" lớn nhất.
4. Xác định cận `lo = 1` (không phải 0 — tránh chia cho 0), `hi = max(quantities)`, và cận dưới `⌈Σq / n⌉`.
5. Tránh nhầm với "chia đều trung bình" (`⌈Σq/n⌉` chưa chắc khả thi vì mỗi cửa hàng chỉ nhận **một loại**).

## 2. Đề bài & ví dụ

Có `n` cửa hàng và `m` loại sản phẩm; `quantities[i]` là số sản phẩm loại `i`. Phân phối **toàn bộ** sản phẩm:
- Mỗi cửa hàng nhận **tối đa một loại** sản phẩm (số lượng tùy ý, có thể 0).
- Gọi `x` là số sản phẩm lớn nhất mà một cửa hàng nhận. Hãy làm `x` **nhỏ nhất**.

Trả về `x` nhỏ nhất có thể.

**Ví dụ 1:** `n = 6, quantities = [11,6]` → `3` (11 chia 4 cửa hàng `2,3,3,3`; 6 chia 2 cửa hàng `3,3`).

**Ví dụ 2:** `n = 7, quantities = [15,10,10]` → `5`.

**Ví dụ 3:** `n = 1, quantities = [100000]` → `100000`.

**Ràng buộc:** `1 ≤ m ≤ n ≤ 10⁵`; `1 ≤ quantities[i] ≤ 10⁵`.

In [1]:
import random
import time
import heapq
import bisect
from typing import List

examples = [((6, [11, 6]), 3), ((7, [15, 10, 10]), 5), ((1, [100000]), 100000)]
for (n, q), exp in examples:
    print(f"n={n}, quantities={q} -> kỳ vọng {exp}")

n=6, quantities=[11, 6] -> kỳ vọng 3
n=7, quantities=[15, 10, 10] -> kỳ vọng 5
n=1, quantities=[100000] -> kỳ vọng 100000


## 3. Kiến thức nền

### 3.1 Số cửa hàng tối thiểu cho một loại

Nếu mỗi cửa hàng nhận tối đa `x` sản phẩm, loại có `q` sản phẩm cần ít nhất `⌈q / x⌉` cửa hàng (và đủ với số đó). `⌈q/x⌉ = (q + x − 1) // x`.

In [2]:
for q, x in [(11, 3), (6, 3), (11, 4), (15, 5), (10, 3)]:
    print(f"q={q}, x={x}: cần {(q + x - 1) // x} cửa hàng")

q=11, x=3: cần 4 cửa hàng
q=6, x=3: cần 2 cửa hàng
q=11, x=4: cần 3 cửa hàng
q=15, x=5: cần 3 cửa hàng
q=10, x=3: cần 4 cửa hàng


### 3.2 Tính đơn điệu

`x` càng lớn thì `⌈q/x⌉` càng nhỏ (không tăng) ⇒ `need(x) = Σ ⌈q_i/x⌉` không tăng ⇒ `ok(x) = [need(x) ≤ n]` có dạng `sai…sai, đúng…đúng`.

### 3.3 Binary search tìm `x` nhỏ nhất thỏa

Duy trì `lo` (chưa chắc thỏa) và `hi` (chắc chắn thỏa), thu hẹp cho tới khi `lo = hi`.

In [3]:
def need(q, x):
    return sum((v + x - 1) // x for v in q)

q = [11, 6]
print([(x, need(q, x), need(q, x) <= 6) for x in range(1, 8)])

[(1, 17, False), (2, 9, False), (3, 6, True), (4, 5, True), (5, 5, True), (6, 3, True), (7, 3, True)]


### 3.4 Cận của đáp án

- `x ≥ 1` (mỗi loại có ít nhất 1 sản phẩm).
- `x ≤ max(q)`: mỗi loại một cửa hàng là khả thi vì `m ≤ n`.
- `x ≥ ⌈Σq / n⌉`: `n` cửa hàng, mỗi cửa hàng `≤ x` ⇒ `n·x ≥ Σq`.

## 4. Ý tưởng nền tảng / trực giác

Thay vì hỏi "chia thế nào", hỏi ngược: "**với giới hạn `x`, có chia được không?**" — câu hỏi này dễ: mỗi loại chiếm `⌈q/x⌉` cửa hàng, các loại không dùng chung cửa hàng ⇒ cần `Σ⌈q/x⌉` cửa hàng. Đáp án là `x` nhỏ nhất mà số này `≤ n`.

Tính tay **Ví dụ 2** (`n = 7`, `[15, 10, 10]`):

| x | ⌈15/x⌉ | ⌈10/x⌉ | ⌈10/x⌉ | tổng | ≤ 7? |
|---|---|---|---|---|---|
| 3 | 5 | 4 | 4 | 13 | ✗ |
| 4 | 4 | 3 | 3 | 10 | ✗ |
| 5 | 3 | 2 | 2 | 7 | **✓** |
| 6 | 3 | 2 | 2 | 7 | ✓ |

Cận dưới trung bình `⌈35/7⌉ = 5` ở đây tình cờ khả thi.

In [4]:
n, q = examples[1][0]
for x in range(3, 7):
    print(f"x={x}: cần {[(v + x - 1) // x for v in q]} -> {need(q, x)} cửa hàng, khả thi: {need(q, x) <= n}")

x=3: cần [5, 4, 4] -> 13 cửa hàng, khả thi: False
x=4: cần [4, 3, 3] -> 10 cửa hàng, khả thi: False
x=5: cần [3, 2, 2] -> 7 cửa hàng, khả thi: True
x=6: cần [3, 2, 2] -> 7 cửa hàng, khả thi: True


## 5. Cách 1 — Brute Force / baseline: thử `x = 1, 2, 3, …`

In [5]:
def minmax_bruteforce(n, quantities):
    x = 1
    while need(quantities, x) > n:
        x += 1
    return x

for (n, q), exp in examples:
    got = minmax_bruteforce(n, q); print(got, "kỳ vọng", exp); assert got == exp

3 kỳ vọng 3
5 kỳ vọng 5
100000 kỳ vọng 100000


**Độ phức tạp:** `O(x* · m)` — tệ nhất `10⁵ · 10⁵ = 10¹⁰` (Ví dụ 3 cần `10⁵` lần thử). Có thể bắt đầu từ cận dưới `⌈Σq/n⌉` (Cách 4) để nhanh hơn trong thực tế.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (đảo câu hỏi):** "tối ưu" ⇒ "kiểm tra khả thi" + tính đơn điệu ⇒ binary search.

**Insight 2 (kiểm tra `O(m)`):** `ok(x) = Σ ⌈q_i/x⌉ ≤ n`.

**Insight 3 (tham lam tương đương):** bắt đầu mỗi loại một cửa hàng; còn `n − m` cửa hàng thừa, mỗi lần cấp một cửa hàng cho loại có `⌈q_i / k_i⌉` (tải hiện tại) **lớn nhất**. Sau khi cấp hết, đáp án là tải lớn nhất. Dùng max-heap.

**Invariant (binary search):** `ok(hi)` luôn đúng; mọi `x < lo` đều không khả thi.

### 6.1 Minh họa: trung bình không phải lúc nào cũng khả thi

Với `n = 3, quantities = [5, 1]` (ví dụ tự dựng, đối chiếu lại đề gốc): trung bình `⌈6/3⌉ = 2`, nhưng loại 2 chỉ có 1 sản phẩm vẫn chiếm trọn một cửa hàng ⇒ loại 1 chỉ có 2 cửa hàng ⇒ `x = ⌈5/2⌉ = 3`.

In [6]:
n, q = 3, [5, 1]
print("trung bình:", -(-sum(q) // n), "| need(2) =", need(q, 2), "> n | đáp án:", minmax_bruteforce(n, q))

trung bình: 2 | need(2) = 4 > n | đáp án: 3


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Binary search trên `x` (bản nộp)

In [7]:
class Solution:
    def minimizedMaximum(self, n: int, quantities: List[int]) -> int:
        lo, hi = 1, max(quantities)
        while lo < hi:
            mid = (lo + hi) // 2
            if sum((q + mid - 1) // mid for q in quantities) <= n:
                hi = mid
            else:
                lo = mid + 1
        return lo

print([Solution().minimizedMaximum(*a) for a, _ in examples])

[3, 5, 100000]


**Độ phức tạp Cách 2:** `O(m log max q)` ≈ `17 · 10⁵`.

### 7.2 Cách 3 — Tham lam với max-heap

In [8]:
def minmax_heap(n, quantities):
    heap = [(-q, q, 1) for q in quantities]           # (-tải, tổng, số cửa hàng)
    heapq.heapify(heap)
    for _ in range(n - len(quantities)):
        _, q, k = heapq.heappop(heap)
        k += 1
        heapq.heappush(heap, (-((q + k - 1) // k), q, k))
    return -heap[0][0]

print([minmax_heap(*a) for a, _ in examples])

[3, 5, 100000]


Mỗi cửa hàng thừa được trao cho loại đang là "nút thắt" (tải lớn nhất); trao cho loại khác không giảm được max. Ta sẽ chứng minh ở mục 10 rằng kết quả bằng đáp án của binary search.

**Độ phức tạp Cách 3:** `O(m + (n − m) log m)`.

### 7.3 Cách 4 — Quét tăng dần từ cận dưới `⌈Σq/n⌉`

In [9]:
def minmax_from_lower(n, quantities):
    x = max(1, -(-sum(quantities) // n))
    while sum((q + x - 1) // x for q in quantities) > n:
        x += 1
    return x

print([minmax_from_lower(*a) for a, _ in examples])

[3, 5, 100000]


Thường chỉ cần vài bước vì cận dưới khá sát; nhưng tệ nhất (nhiều loại nhỏ, một loại lớn) có thể phải tăng nhiều lần — xem mục 11.

## 8. TRACE từng bước

Trace Cách 2 trên **Ví dụ 1** (`n = 6`, `[11, 6]`):

In [10]:
def trace(n, q):
    lo, hi = 1, max(q)
    while lo < hi:
        mid = (lo + hi) // 2; nd = need(q, mid)
        print(f"lo={lo:>2}, hi={hi:>2}, mid={mid:>2}: cần {nd:>2} cửa hàng -> {'khả thi, hi = mid' if nd <= n else 'không, lo = mid + 1'}")
        if nd <= n: hi = mid
        else: lo = mid + 1
    print("=> x =", lo); return lo

assert trace(6, [11, 6]) == 3

lo= 1, hi=11, mid= 6: cần  3 cửa hàng -> khả thi, hi = mid
lo= 1, hi= 6, mid= 3: cần  6 cửa hàng -> khả thi, hi = mid
lo= 1, hi= 3, mid= 2: cần  9 cửa hàng -> không, lo = mid + 1
=> x = 3


| lo | hi | mid | ⌈11/mid⌉ + ⌈6/mid⌉ | khả thi? |
|---|---|---|---|---|
| 1 | 11 | 6 | 2 + 1 = 3 | ✓ ⇒ hi = 6 |
| 1 | 6 | 3 | 4 + 2 = 6 | ✓ ⇒ hi = 3 |
| 1 | 3 | 2 | 6 + 3 = 9 | ✗ ⇒ lo = 3 |

Kết thúc `lo = hi = 3`.

## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 2** — binary search trên đáp án.

In [11]:
class Solution:
    def minimizedMaximum(self, n: int, quantities: List[int]) -> int:
        lo, hi = 1, max(quantities)
        while lo < hi:
            mid = (lo + hi) // 2
            if sum((q + mid - 1) // mid for q in quantities) <= n:
                hi = mid
            else:
                lo = mid + 1
        return lo

for (n, q), exp in examples:
    got = Solution().minimizedMaximum(n, q); print(got, exp); assert got == exp

3 3
5 5
100000 100000


## 10. Vì sao thuật toán đúng

**Khả thi ⇔ `Σ⌈q_i/x⌉ ≤ n`.** (⇒) Trong một phân phối có max `≤ x`, loại `i` được chia cho các cửa hàng **riêng**, mỗi cửa hàng `≤ x` ⇒ ít nhất `⌈q_i/x⌉` cửa hàng; các loại không chung cửa hàng ⇒ tổng `≤ n`. (⇐) Chia loại `i` thành `⌈q_i/x⌉` phần mỗi phần `≤ x`, dùng `Σ⌈q_i/x⌉ ≤ n` cửa hàng; cửa hàng còn lại nhận 0.

**Đơn điệu** (§3.2) ⇒ binary search trên `[1, max q]` (với `ok(max q)` đúng vì `m ≤ n`) trả `x` nhỏ nhất khả thi.

**Tham lam heap.** Với `k_i` cửa hàng cho loại `i`, max tải tốt nhất là `max_i ⌈q_i/k_i⌉`. Ta cần phân `n` cửa hàng (`k_i ≥ 1`, `Σk_i = n`) để cực tiểu max. Giả sử đáp án tối ưu là `x*`; mọi phân phối đạt `x*` có `k_i ≥ ⌈q_i/x*⌉`. Tham lam luôn cấp cửa hàng cho loại có tải lớn nhất; nếu tải lớn nhất hiện tại `> x*` thì loại đó có `k_i < ⌈q_i/x*⌉` ⇒ "thiếu", và tổng các phần thiếu `≤ n − Σk` (vì `Σ⌈q/x*⌉ ≤ n`). Mỗi bước giảm tổng thiếu 1 ⇒ khi hết cửa hàng, không còn loại nào tải `> x*`. ∎

### 10.1 Kiểm chứng thực nghiệm

So bốn cách trên dữ liệu ngẫu nhiên, và kiểm tra cận `⌈Σq/n⌉ ≤ x ≤ max q`.

In [12]:
rng = random.Random(1010)
for _ in range(2000):
    m = rng.randint(1, 6); n = rng.randint(m, m + 12); q = [rng.randint(1, rng.choice([5, 30, 200])) for _ in range(m)]
    ref = minmax_bruteforce(n, q)
    assert ref == Solution().minimizedMaximum(n, q) == minmax_heap(n, q) == minmax_from_lower(n, q)
    assert -(-sum(q) // n) <= ref <= max(q)
print("4 cách khớp nhau; cận ⌈Σq/n⌉ ≤ x ≤ max q (2000 bộ)")

4 cách khớp nhau; cận ⌈Σq/n⌉ ≤ x ≤ max q (2000 bộ)


## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Thử từ 1 | `O(x* · m)` | `O(1)` |
| 2. Binary search | `O(m log max q)` | `O(1)` |
| 3. Heap | `O((n − m) log m)` | `O(m)` |
| 4. Từ cận dưới | `O((x* − ⌈Σq/n⌉ + 1) · m)` | `O(1)` |

In [13]:
rng = random.Random(2064)
cases = [(10 ** 5, [rng.randint(1, 10 ** 5) for _ in range(10 ** 5)]),
         (10 ** 5, [10 ** 5] + [1] * (10 ** 5 - 1)),
         (2 * 10 ** 4, [rng.randint(1, 10 ** 5) for _ in range(10 ** 4)])]
for name, f in [("bs", Solution().minimizedMaximum), ("heap", minmax_heap)]:
    t = time.perf_counter(); r = [f(n, q) for n, q in cases]; print(f"{name:<10}: {r} ({(time.perf_counter() - t) * 1000:.0f} ms)")
for n, q in cases:
    lb = -(-sum(q) // n); ans = Solution().minimizedMaximum(n, q)
    print(f"from_lower cần thử {ans - lb + 1:>6} giá trị x (mỗi lần O(m) với m = {len(q)}) — cận dưới {lb}, đáp án {ans}")

bs        : [100000, 100000, 33638] (136 ms)


heap      : [100000, 100000, 33638] (44 ms)
from_lower cần thử  49925 giá trị x (mỗi lần O(m) với m = 100000) — cận dưới 50076, đáp án 100000
from_lower cần thử  99999 giá trị x (mỗi lần O(m) với m = 100000) — cận dưới 2, đáp án 100000
from_lower cần thử   8458 giá trị x (mỗi lần O(m) với m = 10000) — cận dưới 25181, đáp án 33638


Ca thứ hai: `n = m` ⇒ mỗi loại đúng một cửa hàng ⇒ `x = 10⁵`, trong khi cận dưới chỉ là `2` ⇒ Cách 4 phải thử gần `10⁵` giá trị, mỗi lần `O(m) = 10⁵` ⇒ `10¹⁰` phép (không chạy ở đây). Binary search không bị ảnh hưởng.

## 12. Bộ test

In [14]:
ALL = {"bs": Solution().minimizedMaximum, "heap": minmax_heap, "from_lower": minmax_from_lower}
def run_all(n, q, brute=True):
    res = {k: f(n, list(q)) for k, f in ALL.items() if brute or k != "from_lower"}
    if brute:
        res["brute"] = minmax_bruteforce(n, q)
    assert len(set(res.values())) == 1, (n, q, res)
    return res["bs"]

for (n, q), exp in examples:
    assert run_all(n, q) == exp
# ví dụ tự dựng (đối chiếu lại đề gốc) — kiểm bằng vét cạn trong run_all
edge = [((3, [5, 1]), 3), ((1, [1]), 1), ((5, [5]), 1), ((4, [5]), 2), ((2, [3, 3]), 3), ((10, [1, 1]), 1), ((3, [7]), 3)]
for (n, q), exp in edge:
    assert run_all(n, q) == exp, (n, q, run_all(n, q))
rng = random.Random(20640)
for _ in range(500):
    m = rng.randint(1, 8); n = rng.randint(m, 30); run_all(n, [rng.randint(1, 100) for _ in range(m)])
for _ in range(3):
    m = rng.randint(1, 5000); n = rng.randint(m, 10 ** 5); run_all(n, [rng.randint(1, 10 ** 5) for _ in range(m)], brute=False)
print("Ví dụ, 7 edge case, 500 bộ nhỏ, 3 bộ lớn: OK")

Ví dụ, 7 edge case, 500 bộ nhỏ, 3 bộ lớn: OK


### 12.1 Kiểm tra tính chất

- Tăng `n` không làm đáp án tăng; tăng một `q_i` không làm đáp án giảm.
- Hoán vị `quantities` không đổi đáp án.
- Với `x = đáp án`, `need(x) ≤ n` và (nếu `x > 1`) `need(x − 1) > n`.

In [15]:
rng = random.Random(121)
f = Solution().minimizedMaximum
for _ in range(1000):
    m = rng.randint(1, 6); n = rng.randint(m, 25); q = [rng.randint(1, 60) for _ in range(m)]; a = f(n, q)
    assert f(n + 1, q) <= a
    i = rng.randrange(m); q2 = q[:]; q2[i] += rng.randint(1, 10); assert f(n, q2) >= a
    sh = q[:]; rng.shuffle(sh); assert f(n, sh) == a
    assert need(q, a) <= n and (a == 1 or need(q, a - 1) > n)
print("4 tính chất đúng trên 1000 bộ")

4 tính chất đúng trên 1000 bộ


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Trả về trung bình `⌈Σq / n⌉`

In [16]:
def wrong_average(n, quantities):
    return -(-sum(quantities) // n)                              # SAI

for (n, q), exp in examples + [((3, [5, 1]), 3)]:
    print(f"n={n}, q={q}: sai {wrong_average(n, q)} | đúng {exp}")

n=6, q=[11, 6]: sai 3 | đúng 3
n=7, q=[15, 10, 10]: sai 5 | đúng 5
n=1, q=[100000]: sai 100000 | đúng 100000
n=3, q=[5, 1]: sai 2 | đúng 3


**Vì sao sai:** mỗi cửa hàng chỉ nhận **một loại** ⇒ không thể "trộn" phần dư của các loại. `⌈Σq/n⌉` chỉ là cận dưới. **Sửa:** kiểm tra khả thi bằng `Σ⌈q/x⌉`.

### Lỗi 2 — Dùng `q // x` (làm tròn xuống) trong hàm kiểm tra

In [17]:
def wrong_floor(n, quantities):
    lo, hi = 1, max(quantities)
    while lo < hi:
        mid = (lo + hi) // 2
        if sum(q // mid for q in quantities) <= n: hi = mid           # SAI: phải làm tròn lên
        else: lo = mid + 1
    return lo

for (n, q), exp in examples:
    print(f"n={n}, q={q}: sai {wrong_floor(n, q)} | đúng {exp}")

n=6, q=[11, 6]: sai 3 | đúng 3
n=7, q=[15, 10, 10]: sai 4 | đúng 5
n=1, q=[100000]: sai 50001 | đúng 100000


**Vì sao sai:** 11 sản phẩm với giới hạn 2 cần 6 cửa hàng (`⌈11/2⌉`), không phải 5 — phần dư cũng cần cửa hàng. **Sửa:** `(q + x − 1) // x`.

### Lỗi 3 — `lo = 0` (chia cho 0)

In [18]:
def wrong_lo_zero(n, quantities):
    lo, hi = 0, max(quantities)
    try:
        while lo < hi:
            mid = (lo + hi) // 2
            if sum((q + mid - 1) // mid for q in quantities) <= n: hi = mid
            else: lo = mid + 1
        return lo
    except ZeroDivisionError as e:
        return f"ZeroDivisionError: {e}"

print(wrong_lo_zero(1, [1]), "| (tự dựng: n = 1, quantities = [1]) đúng:", Solution().minimizedMaximum(1, [1]))

ZeroDivisionError: integer division or modulo by zero | (tự dựng: n = 1, quantities = [1]) đúng: 1


**Vì sao sai:** với `hi = 1`, `mid = 0` ⇒ chia cho 0. Đáp án luôn `≥ 1`. **Sửa:** `lo = 1`.

### Lỗi 4 — Chia cửa hàng theo tỉ lệ rồi làm tròn

In [19]:
def wrong_proportional(n, quantities):
    S = sum(quantities)
    k = [max(1, round(q * n / S)) for q in quantities]             # SAI: có thể vượt n hoặc lệch tối ưu
    return max((q + ki - 1) // ki for q, ki in zip(quantities, k)), sum(k)

for (n, q) in [(6, [11, 6]), (7, [15, 10, 10]), (4, [1, 1, 10]), (5, [9, 1, 1])]:
    ans, used = wrong_proportional(n, q)
    print(f"n={n}, q={q}: sai {ans} (dùng {used} cửa hàng) | đúng {Solution().minimizedMaximum(n, q)}")

n=6, q=[11, 6]: sai 3 (dùng 6 cửa hàng) | đúng 3
n=7, q=[15, 10, 10]: sai 5 (dùng 7 cửa hàng) | đúng 5
n=4, q=[1, 1, 10]: sai 4 (dùng 5 cửa hàng) | đúng 5
n=5, q=[9, 1, 1]: sai 3 (dùng 6 cửa hàng) | đúng 3


**Vì sao sai:** làm tròn tỉ lệ có thể dùng **quá** `n` cửa hàng (phân phối không hợp lệ) hoặc phân bổ chưa tối ưu. **Sửa:** binary search hoặc heap.

### Lỗi 5 — Cập nhật sai hướng trong binary search (`hi = mid − 1`)

In [20]:
def wrong_bs(n, quantities):
    lo, hi = 1, max(quantities)
    while lo < hi:
        mid = (lo + hi) // 2
        if sum((q + mid - 1) // mid for q in quantities) <= n: hi = mid - 1     # SAI: loại mất mid khả thi
        else: lo = mid + 1
    return lo

for (n, q), exp in examples:
    print(f"n={n}, q={q}: sai {wrong_bs(n, q)} | đúng {exp}")

n=6, q=[11, 6]: sai 2 | đúng 3
n=7, q=[15, 10, 10]: sai 5 | đúng 5
n=1, q=[100000]: sai 100000 | đúng 100000


**Vì sao sai:** khi `mid` khả thi, nó có thể **chính là** đáp án; `hi = mid − 1` bỏ mất nó. **Sửa:** `hi = mid` (giữ bất biến "`hi` khả thi").

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Minimize the maximum" / "maximize the minimum" ⇒ **binary search trên đáp án**.
- Viết `ok(x)` trước; kiểm tra tính đơn điệu; chọn cận `[lo, hi]` có `ok(hi)` đúng.
- Chia `q` vào các phần `≤ x` ⇒ `⌈q/x⌉` phần.
- Tham lam heap "cấp thêm tài nguyên cho nút thắt" thường tương đương binary search.

## 15. Biến thể của bài

1. **875 Koko Eating Bananas**: `Σ⌈p/k⌉ ≤ h` — gần như giống hệt.
2. **1760 Minimum Limit of Balls in a Bag**: `Σ(⌈q/x⌉ − 1) ≤ maxOperations`.
3. **410 Split Array Largest Sum**: kiểm tra bằng tham lam chia đoạn liên tiếp.
4. **Mỗi cửa hàng nhận tối đa `t` loại** — kiểm tra phức tạp hơn.
5. **Cửa hàng có sức chứa khác nhau** — binary search + ghép cặp.

### 15.1 Code biến thể: 1760 Minimum Limit of Balls in a Bag

In [21]:
def minimum_size(nums, max_ops):
    lo, hi = 1, max(nums)
    while lo < hi:
        mid = (lo + hi) // 2
        if sum((x - 1) // mid for x in nums) <= max_ops: hi = mid      # ⌈x/mid⌉ − 1 = (x − 1)//mid
        else: lo = mid + 1
    return lo

rng = random.Random(1760)
for _ in range(500):
    nums = [rng.randint(1, 40) for _ in range(rng.randint(1, 5))]; ops = rng.randint(0, 15)
    assert minimum_size(nums, ops) == minmax_bruteforce(len(nums) + ops, nums)    # ops phép chia = thêm ops "cửa hàng"
print("1760 = bài này với n = len(nums) + maxOperations;", minimum_size([9], 2), minimum_size([2, 4, 8, 2], 4))

1760 = bài này với n = len(nums) + maxOperations; 3 2


Nhận xét đẹp: mỗi phép chia túi tạo thêm một túi ⇒ sau `k` phép có `len(nums) + k` túi, đúng là bài 2064 với `n = len(nums) + maxOperations`.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử từ 1 | Vét cạn `x` | `O(x*·m)` | `O(1)` | Kiểm chứng |
| 2. Binary search | `Σ⌈q/x⌉ ≤ n` | `O(m log Q)` | `O(1)` | **Bản nộp** |
| 3. Heap | Cấp cho nút thắt | `O(n log m)` | `O(m)` | Cần cả cách phân bổ |
| 4. Từ cận dưới | Quét tăng | phụ thuộc dữ liệu | `O(1)` | Khi cận dưới sát |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `n = 5, [5]`; (b) `n = 4, [5]`; (c) `n = 2, [3,3]`; (d) `n = 10, [1,1]`.

**Bài 2 (tính tay):** Với `n = 5, quantities = [9, 4]`, lập bảng `need(x)` cho `x = 1..5` và đáp án.

**Bài 3 (điền TODO):** Hoàn thiện `feasible(n, quantities, x)`.

**Bài 4 (tự viết lại):** Viết `distribution(n, quantities)` trả danh sách `n` số (số sản phẩm mỗi cửa hàng, cửa hàng nào cũng chỉ một loại) đạt max tối ưu.

**Bài 5 (mini challenge):** Mỗi cửa hàng `j` có **sức chứa** `cap[j]` (không được vượt). Kiểm tra có phân phối được toàn bộ sản phẩm không (mỗi cửa hàng một loại), và viết vét cạn cho `n ≤ 6` để đối chiếu một chiến lược tham lam của bạn.

In [22]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def feasible(n, quantities, x):
    # TODO: trả sum((q + x - 1) // x for q in quantities) <= n
    pass

In [23]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def distribution(n, quantities):
    # TODO: x = đáp án; mỗi loại chia thành ceil(q/x) phần gần đều; bù 0 cho cửa hàng thừa
    pass

def feasible_caps(caps, quantities):
    # TODO: vét cạn gán mỗi cửa hàng cho một loại (hoặc bỏ trống) và kiểm tra tổng sức chứa mỗi loại >= q
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: `need(x) ≤ n`. Mức 2: `x ≥ 1`. Mức 3: (a) `1`; (b) `2`; (c) `3` (mỗi loại một cửa hàng); (d) `1`.

**Bài 2.** Mức 1: `need(x) = ⌈9/x⌉ + ⌈4/x⌉`. Mức 2: `x = 1..5`: `13, 7, 5, 4, 3`. Mức 3: `x = 3` là nhỏ nhất với `need ≤ 5`.

**Bài 3.** Mức 1: làm tròn lên. Mức 2: cộng qua mọi loại. Mức 3: so với `n`.

**Bài 4.** Mức 1: `k = ⌈q/x⌉` phần cho loại `q`. Mức 2: chia gần đều: `q // k` và `q % k` phần thêm 1. Mức 3: số phần tổng `≤ n`, thêm số 0.

**Bài 5.** Mức 1: tổng sức chứa các cửa hàng gán cho loại `i` phải `≥ q_i`. Mức 2: vét cạn `(m + 1)^n` cách gán. Mức 3: tham lam (ví dụ cấp cửa hàng lớn nhất cho loại thiếu nhiều nhất) không phải luôn đúng — hãy tìm phản ví dụ bằng vét cạn.

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1, 2, 3
f = Solution().minimizedMaximum
b1 = [f(5, [5]), f(4, [5]), f(2, [3, 3]), f(10, [1, 1])]
print("Bài 1:", b1); assert b1 == [1, 2, 3, 1]
b2 = [need([9, 4], x) for x in range(1, 6)]
print("Bài 2:", b2, "->", f(5, [9, 4])); assert b2 == [13, 7, 5, 4, 3] and f(5, [9, 4]) == 3
def feasible(n, quantities, x):
    return sum((q + x - 1) // x for q in quantities) <= n
assert feasible(6, [11, 6], 3) and not feasible(6, [11, 6], 2)
print("Bài 3: OK")

Bài 1: [1, 2, 3, 1]
Bài 2: [13, 7, 5, 4, 3] -> 3
Bài 3: OK


In [25]:
# Bài 4, 5
from itertools import product
def distribution(n, quantities):
    x = f(n, quantities); out = []
    for q in quantities:
        k = (q + x - 1) // x
        out += [q // k + (1 if t < q % k else 0) for t in range(k)]
    return out + [0] * (n - len(out))

def feasible_caps(caps, quantities):
    m = len(quantities)
    for assign in product(range(m + 1), repeat=len(caps)):          # m = để trống
        tot = [0] * m
        for j, a in enumerate(assign):
            if a < m: tot[a] += caps[j]
        if all(tot[i] >= quantities[i] for i in range(m)):
            return True
    return False

def greedy_caps(caps, quantities):
    rem = sorted(quantities, reverse=True); caps = sorted(caps, reverse=True)
    import heapq as hq
    h = [-q for q in quantities]; hq.heapify(h)
    for c in caps:
        if not h: break
        q = -hq.heappop(h) - c
        if q > 0: hq.heappush(h, -q)
    return not h

rng = random.Random(5)
mismatch = None
for _ in range(400):
    m = rng.randint(1, 6); n = rng.randint(m, 12); q = [rng.randint(1, 40) for _ in range(m)]
    d = distribution(n, q); assert len(d) == n and sum(d) == sum(q) and max(d) == f(n, q)
    caps = [rng.randint(1, 8) for _ in range(rng.randint(1, 5))]; qq = [rng.randint(1, 10) for _ in range(rng.randint(1, 3))]
    if feasible_caps(caps, qq) != greedy_caps(caps, qq) and mismatch is None: mismatch = (caps, qq, feasible_caps(caps, qq))
print("Bài 4: OK |", distribution(6, [11, 6]))
print("Bài 5: phản ví dụ cho tham lam (caps, quantities, đáp án đúng):", mismatch)

Bài 4: OK | [3, 3, 3, 2, 3, 3]
Bài 5: phản ví dụ cho tham lam (caps, quantities, đáp án đúng): ([5, 7, 5, 6, 4], [8, 8, 6], True)


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
lo, hi = 1, max(quantities)
while lo < hi:
    mid = (lo + hi) // 2
    if sum((q + mid - 1) // mid for q in quantities) <= n:   # khả thi
        hi = mid
    else:
        lo = mid + 1
return lo
```

**Checklist:**
- [ ] Làm tròn **lên** `⌈q/x⌉`?
- [ ] `lo = 1` (không chia cho 0)?
- [ ] `hi = max(q)` khả thi (vì `m ≤ n`)?
- [ ] Khi khả thi: `hi = mid` (không `mid − 1`)?
- [ ] Không dùng trung bình làm đáp án?